# 02 — Feature Transformation

**Status:** `PENDING MANUAL EXECUTION`


## 1. Imports


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold


## 2. Project Root & Paths


In [ ]:
def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    raise RuntimeError("Could not locate project root.")

PROJECT_ROOT = find_project_root()
IN_CSV = PROJECT_ROOT / "data" / "interim" / "features" / "feature_engineered_dataset.csv"
TRANS_DIR = PROJECT_ROOT / "data" / "interim" / "features" / "transformation"
TRANS_DIR.mkdir(parents=True, exist_ok=True)


## 3. Load Feature-Engineered Dataset


In [ ]:
df = pd.read_csv(IN_CSV)
print("Loaded shape:", df.shape)


## 4. Separate Target & Compute Duplicate Group IDs


In [ ]:
y = (df["eligibility"].astype(str).str.upper() == "ELIGIBLE").astype(int)
groups = pd.util.hash_pandas_object(df.drop(columns=["eligibility"]), index=False).astype(str)


## 5. Group-Aware Stratified Split


In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, y, groups=groups))


## 6. Verify Split Integrity


In [ ]:
print(f"Train: {len(train_idx)} | Test: {len(test_idx)} | Group overlap: {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}")
